# EDA 004 · Volatility training set

**The question.** How much will this stock's price swing over the next three months?

**One row** is a company at a month end (2012 – September 2022). **The answer**,
`vol_63_future`, is the stock's *volatility* over the next 63 trading days (about three months):
the standard deviation of its daily returns, scaled to a year ("annualised"). A value of 0.40
means a typical one-year swing of about ±40%; a calm large company is near 0.20, a small biotech
can be above 1.00. **The naive baseline** is `vol_63`: the same measure over the past 63 days,
assumed to continue. The model (005) has to beat it.

The answer is always positive and has a long right tail, so it is studied on a log scale, as the
model will predict its logarithm (`config/risk_params.yaml` v5).

**How this notebook works**

- **What it does.** It reads one training set from the private export, keeps the development
  rows, and goes through it section by section: what is in it, who is covered, what is missing,
  what is extreme, what the answer looks like, how well the simple baseline already does, which
  features relate to the answer, which features repeat each other, and whether anything leaks
  from the future. It ends with written findings and advice for the model.
- **Why.** A model is only as good as its data. Problems found here are cheap to fix; found after
  training, they mean training again. The findings also tell the model builder where to start.
- **It changes nothing.** The notebook only reads. A problem that needs fixing is fixed in the
  pipeline (`src/sentinel/`), with tests, so that it applies to every year alike (the holdout
  included) and to the daily scores; it is never patched here.
- **Cleaning already applied by the pipeline** (decided on 30 October 2026, `docs/decisions.md`):
  V2-22, a foreign filer's market value is blank (its US price is often an ADR's while its share
  count is of ordinary shares, so the product would be wrong); V2-23, a share count typed in
  wrongly (zero or below, or 100 times away from the same filing's weighted-average count or,
  without one, from the company's last accepted count) is left out, judged only by what was known
  when it was filed; V2-24, an insider trade line worth more than the whole company has no dollar
  value (a typo in its price or shares). Section 5 checks that these hold.
- **Privacy.** The export holds prices licensed from Tiingo. Outputs are stripped when the
  notebook is committed (the repository is public); the written findings quote only aggregates.
- **Run it** with one command, under the machine's memory cap (about 30 seconds):
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/<this notebook>`

**Words used below**

- **Row**: one company on one date, with everything known about it at that day's close (the
  *features*) and what happened next (the *answer*).
- **Eligible**: the rows a model may learn from: the company filed something with the SEC in the
  last 12 months, is not a fund or blank-check company, its price is at least \$1, it trades at
  least \$10,000 a day, and it has at least 63 days of prices. Per question: price-only stocks
  need not file for volatility; foreign filers are left out of dilution.
- **Development rows**: rows up to 2022 with an answer. Rows from 2023 on are the sealed holdout
  and are never looked at here.
- **Test years**: 2019, 2020, 2021, 2022. Each is predicted by a model trained only on earlier
  years ("walk-forward").
- **Purge**: a training row whose answer window reaches into the test year is dropped, so the
  model never trains on days it is tested on.
- **Spearman correlation (ρ)**: how well two columns rank alike, from −1 to +1; 0 means no order in
  common. It ignores scale, so it is safe with long tails.
- **Decile**: one of ten equal-sized groups after sorting by a feature (decile 1 = the lowest 10%).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentinel import eda

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.4g}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
TEST_YEARS = eda.TEST_YEARS

## 1. Load

**What:** read the file, keep the development rows (dated 2022 or earlier), and count rows,
companies and dates; check that each (company, date) appears once. Then `ok` = the rows a model
will learn from: eligible and with an answer. **Why:** every later section measures relations on
`ok`, as the model will see them; the counts are the scale of everything that follows.

In [ ]:
SET, ANSWER = "volatility", "vol_63_future"
df = eda.load(SET)
FEATURES = eda.features(df)
print(f"{len(df):,} rows, {df['cik'].nunique():,} companies, {df.shape[1]} columns, "
      f"{len(FEATURES)} features")
print("dates:", df["as_of"].min().date(), "to", df["as_of"].max().date())
print("duplicate (cik, as_of) keys:", int(df.duplicated(["cik", "as_of"]).sum()))
ELIG = eda.ELIGIBLE[SET]  # this question's eligibility (V2-26, V2-27)
ok = df[df[ELIG] & df[ANSWER].notna()].copy()  # what the model learns from
print(f"eligible with an answer: {len(ok):,} rows ({len(ok) / len(df):.1%}), "
      f"{ok['cik'].nunique():,} companies")

## 2. Columns by family

**What:** sort the about 100 features into families (price, trading, size, financials, filing
events, insider, short volume, macro) and list what is not a feature (identifiers, flags, the
answer, dates). **Why:** families make 100 columns readable, show which kinds of information the
model has, and make sure nothing that is not a feature (the answer, its window) is used as one.

In [ ]:
fam = pd.Series({c: eda.family(c) for c in FEATURES}, name="family")
display(fam.value_counts().rename("features").to_frame())
display(fam.groupby(fam).apply(lambda s: ", ".join(s.index)).rename("columns").to_frame())
others = [c for c in df.columns if c not in FEATURES]
print("not features:", ", ".join(others))
display(df.dtypes.astype(str).value_counts().rename("columns").to_frame())

## 3. Coverage

Who is in the data, per year and per kind of company. *Coverage level*: `full` (files with the
SEC and has prices), `foreign` (a foreign filer: 20-F or 40-F reports), `price_only` (trades in
the US, files nothing periodic). Size bands by market value: micro below \$300 M, small to
\$2 B, mid to \$10 B, large above; `unknown` where market value is missing.

In [ ]:
year = df["as_of"].dt.year
by_year = eda.coverage_table(df, year, "vol_63_future", ELIG)
display(by_year)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
pd.DataFrame({"eligible": df[df[ELIG]].groupby(year[df[ELIG]]).size(),
              "not eligible": df[~df[ELIG]].groupby(year[~df[ELIG]]).size()}
             ).plot.bar(stacked=True, ax=ax[0], title="rows per year")
by_year["companies"].plot.bar(ax=ax[1], title="companies per year")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["coverage", "filer_type", "foreign_filer", "sector"]:
    display(eda.coverage_table(df, col, "vol_63_future", ELIG).sort_values("rows", ascending=False))
df["size_band"] = eda.size_band(df["market_cap"])
ok["size_band"] = eda.size_band(ok["market_cap"])
display(eda.coverage_table(df, "size_band", "vol_63_future", ELIG))

**Why rows are not eligible.** A row can fail several of the five rules at once. Each question
then has its own flag (decided 30 October 2026): volatility also admits price-only stocks that
pass the price rules (V2-26: volatility needs only prices); dilution leaves out foreign filers
(V2-27: most have no answer); the results move uses the general flag. The two tables after the
reasons compare the general flag with this question's. **Why it
matters:** the model never sees these rows, so whatever they have in common (tiny, untraded,
not filing) is outside what it can forecast. `no reason found` must be 0: otherwise the flag and
the rules disagree.

In [ ]:
display(eda.eligibility_reasons(df))
print(f"this question uses `{ELIG}`:")
display(pd.crosstab(df["eligible"], df[ELIG], margins=True))
display(pd.crosstab(df["coverage"], df[ELIG]))

## 4. Missing values

Share of eligible rows with an answer where each feature is missing. A missing value is not
an error when the fact does not exist for that company (constitution II: missing, never zero):
foreign filers do not file Form 4 or 8-K; companies without financial filings have no ratios; a
ratio with a zero denominator is left empty.

In [ ]:
miss = eda.missing_table(ok, FEATURES)
print(f"features with no missing values: {(miss['missing'] == 0).sum()}; "
      f"over 50% missing: {(miss['missing'] > 0.5).sum()}")
display(miss[miss["missing"] > 0])
display(miss.groupby("family")["missing"].agg(["mean", "max"]).sort_values("mean"))

**Where the gaps come from.** The 15 most-missing features by filer type, coverage level and
year. **Why:** a gap explained by the kind of company (a foreign filer has no Form 4; a company
that does no research has no R&D) is information, not an error; a gap that grows or shrinks by
year would show a change in the data, not in the companies.

In [ ]:
top = miss.index[:15].tolist()
display(ok.groupby("filer_type")[top].apply(lambda g: g.isna().mean()).T)
display(ok.groupby("coverage")[top].apply(lambda g: g.isna().mean()).T)
display(eda.missing_by_year(ok, top))

## 5. Impossible and extreme values

`infinite`: values of ±∞ (a division by zero that slipped through); `negative`: values below
zero where that is impossible (counts, prices, volatilities, amounts). Then each feature's spread:
the 1st percentile (p1), median and 99th percentile (p99) against its minimum and maximum: a
maximum far beyond p99 means a long tail or a data error.

In [ ]:
ext = eda.extremes_table(ok, FEATURES)
bad = ext[(ext["infinite"] > 0) | (ext["negative"] > 0)]
print("features with infinite or impossible negative values:", len(bad))
display(bad)
spread = ext.assign(max_over_p99=ext["max"] / ext["p99"].where(ext["p99"] > 0))
display(spread.sort_values("max_over_p99", ascending=False).head(25))

**Suspect values.** Specific checks behind the long tails above. **Why:** a long tail can be
real (a stock that rose tenfold) or a data error; these checks count the known kinds of error.
The first rows check the pipeline's cleaning rules (V2-22 to V2-24): they should be 0 or close
to it. A foreign filer's market value
should be missing (its share count is of ordinary shares while its US price is often an ADR's,
risk_params v5); no US company was worth \$5 trillion before 2023 (Apple's peak was about \$3
trillion); a 12-month insider total above \$100 billion is a typo in a filing.

In [ ]:
checks = {
    "foreign filer with a market value": ok["foreign_filer"] & ok["market_cap"].notna(),
    "market value 0": ok["market_cap"] == 0,
    "market value above 5 trillion dollars": ok["market_cap"] > 5e12,
    "shares outstanding above 50 billion": ok["shares_outstanding"] > 5e10,
    "one-month return above +1,000%": ok["ret_1m"] > 10,
    "12-month return above +2,000%": ok["ret_12_1"] > 20,
    "insider sales above 100 billion dollars in 12 months": ok["sell_value_12m"] > 1e11,
    "insider buys above 10 billion dollars in 12 months": ok["buy_value_12m"] > 1e10,
    "cash above total assets": ok["cash_to_assets"] > 1,
    "negative leverage": ok["leverage"] < 0,
    "negative book value (book_to_market < 0)": ok["book_to_market"] < 0,
    "results overdue by more than a year": ok["days_to_next_results"] < -365,
}
display(pd.DataFrame({k: {"rows": int(v.sum()), "companies": ok.loc[v, "cik"].nunique(),
                          "share": v.mean()} for k, v in checks.items()}).T)

In [ ]:
print("answers <= 0:", int((ok[ANSWER] <= 0).sum()), "| baseline vol_63 == 0:", int((ok["vol_63"] == 0).sum()))
z = ok[ok[ANSWER] == 0]
print("rows with a zero answer: by year", z["as_of"].dt.year.value_counts().sort_index().to_dict())
print("  their coverage:", z["coverage"].value_counts().to_dict(), "| companies:", z["cik"].nunique())
print("  their median 20-day traded value:", z["addv20"].median(), "| zero-volume share (63 d):", z["zero_volume_63"].median())
for t in [1, 2, 3, 5, 10]:
    print(f"answers above {t:>2} ({t:.0%} a year): {(ok[ANSWER] > t).sum():,}")

## 6. The answer

Its distribution on a log scale, and how its level varies by year (2020's crash shows up),
sector and size.

In [ ]:
display(ok[ANSWER].describe(percentiles=[0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).to_frame())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
np.log10(ok.loc[ok[ANSWER] > 0, ANSWER]).plot.hist(bins=100, ax=ax[0], title="log10 of vol_63_future (> 0)")
q = ok.groupby(ok["as_of"].dt.year)[ANSWER].quantile([0.1, 0.5, 0.9]).unstack()
q.plot(ax=ax[1], title="10th, 50th, 90th percentile by year", marker="o")
plt.tight_layout(); plt.show()
for col in ["sector", "size_band", "coverage", "filer_type"]:
    display(ok.groupby(col)[ANSWER].agg(["count", "median", lambda s: s.quantile(0.9)])
            .rename(columns={"<lambda_0>": "p90"}).sort_values("median"))

## 7. The naive baseline

How well "the last three months continue" predicts the next three. Errors on the log scale:
an error of 0.1 is about 10% too high or too low. `bias` is the mean of log(future / past):
negative means volatility tends to fall back after being measured. `within ±25%`: the share of
rows where the future lies within 25% of the past value.

In [ ]:
b = ok[(ok["vol_63"] > 0) & (ok[ANSWER] > 0)]
err = np.log(b[ANSWER]) - np.log(b["vol_63"])

def skill(e, f):
    return pd.Series({"rows": len(e), "MAE (log)": e.abs().mean(), "RMSE (log)": np.sqrt((e**2).mean()),
                      "bias (log)": e.mean(), "spearman": f["vol_63"].rank().corr(f[ANSWER].rank()),
                      "within ±25%": (e.abs() <= np.log(1.25)).mean()})

years = b["as_of"].dt.year
table = pd.DataFrame({y: skill(err[years == y], b[years == y]) for y in sorted(years.unique())}).T
table.loc["all"] = skill(err, b)
display(table)
print("rows left out (vol_63 missing or 0, or the answer 0):", len(ok) - len(b))

**Mean reversion, as a simple rule.** Volatility that is unusually high tends to fall, and low
volatility tends to rise. A straight line fitted on the log scale, on rows whose window ended
before 2019, shows how strong that pull is: a slope below 1 means "partly back toward normal".
This is a rule-level check for 005, not a chosen model.

In [ ]:
train = b[b["window_end"] < "2019-01-01"]
slope, intercept = np.polyfit(np.log(train["vol_63"]), np.log(train[ANSWER]), 1)
print(f"log(future) = {intercept:.3f} + {slope:.3f} * log(past)  (fitted on {len(train):,} rows)")
test = b[b["as_of"].dt.year.isin(TEST_YEARS)]
pred = intercept + slope * np.log(test["vol_63"])
e_rule = np.log(test[ANSWER]) - pred
e_naive = np.log(test[ANSWER]) - np.log(test["vol_63"])
display(pd.DataFrame({"naive MAE": e_naive.abs().groupby(test["as_of"].dt.year).mean(),
                      "rule MAE": e_rule.abs().groupby(test["as_of"].dt.year).mean()}))

## 8. Feature–answer relations

Spearman ρ of every feature with the future volatility, on eligible rows with an answer; strongest
first. `n` is the rows where the feature is present.

In [ ]:
ranks = eda.rank_table(ok, FEATURES, "vol_63_future")
display(ranks)
display(ranks.groupby("family")["spearman"].apply(lambda s: s.abs().max()).sort_values()
        .rename("strongest |ρ| in family").to_frame())

**The answer by decile** of the ten strongest features (mean of the answer in each tenth).
**Why:** a correlation is one number; the decile curve shows its shape: steady, only at the
extremes, or U-shaped (which a linear model would miss).

In [ ]:
top10 = ranks.index[:10].tolist()
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=True)
for ax, col in zip(axes.flat, top10):
    d = eda.decile_table(ok, col, "vol_63_future")
    ax.plot(d.index.astype(int), d["mean"], marker="o")
    ax.set_title(col, fontsize=9); ax.set_xticks(range(1, 11))
plt.suptitle("mean answer by feature decile (1 = lowest)"); plt.tight_layout(); plt.show()

**Stability over time.** The same ρ within each year for the 15 strongest features. A relation
that flips sign or fades in some years is less trustworthy than its pooled value suggests.

In [ ]:
top15 = ranks.index[:15].tolist()
yearly = eda.rank_by_year(ok, top15, "vol_63_future")
display(yearly)
pooled = ranks.loc[top15, "spearman"]
flips = (np.sign(yearly).ne(np.sign(pooled), axis=0) & yearly.notna()).sum(axis=1)
display(pd.DataFrame({"pooled": pooled, "min": yearly.min(axis=1), "max": yearly.max(axis=1),
                      "years with the other sign": flips}))

**What adds to the baseline.** The ranking above is dominated by past volatility. More useful
for the model: which features relate to the *surprise*, log(future / past), i.e. whether
volatility will rise or fall from where it is.

In [ ]:
b = b.assign(surprise=np.log(b[ANSWER]) - np.log(b["vol_63"]))
surprise = eda.rank_table(b, FEATURES, "surprise")
display(surprise.head(25))

## 9. Redundant features

Pairs of features that rank rows almost identically (|ρ| ≥ 0.9, on a fixed sample of 200,000
rows). Tree models cope with them, but they split importance between the pair and make the model
harder to explain; linear models need one of each pair dropped.

In [ ]:
pairs = eda.redundant_pairs(ok, FEATURES)
print(len(pairs), "pairs")
display(pairs)

## 10. Leakage and purging

*Leakage* is information from after the row's date reaching its features: a model then looks
brilliant in testing and fails live. Checks: (a) every answer window ends after the row date;
(b) no feature relates to the answer implausibly well (|ρ| ≥ 0.95); (c) how many training rows
the purge removes before each test year, and how many remain.

In [ ]:
gap = (ok["window_end"] - ok["as_of"]).dt.days
print("windows ending on or before the row date:", int((gap <= 0).sum()))
display(gap.describe(percentiles=[0.01, 0.5, 0.99]).rename("days from row to window end").to_frame())
print("features with |rho| >= 0.95:", ranks.index[ranks["spearman"].abs() >= 0.95].tolist())
display(eda.purge_table(ok, "window_end"))

## 11. Findings

*Export of 30 October 2026, after the cleaning rules V2-22 to V2-24 (rules V2-28 to V2-32, applied
later that day, changed these numbers only slightly: for volatility 131 fewer eligible rows and
8 fewer answers above 1,000% a year, from placeholder prices), with this question's
eligibility `eligible_volatility` (V2-26: price-only stocks count when they pass the price rules).
Numbers come from the sections named in brackets; all are aggregates.*

### In one paragraph

The volatility set is ready for modelling. 532,144 rows (7,903 companies) are eligible with an
answer, 9,577 of them of price-only stocks; keys are unique, every answer window ends 88–96 days after its row, and no feature is
suspiciously strong. The naive baseline is already good (rank correlation 0.79) but misses by
about 30% on a typical row, and badly in 2020. A one-line *mean reversion* rule beats it in three
of the four test years. The pipeline's cleaning rules hold; a short list of named cases is left
for the validation round.

### What is clean

- **Coverage is steady** [3]: 5,000–7,000 companies a year, about 73% of rows eligible every
  year, 99% of eligible rows have an answer. Price-only stocks: 9,665 of their 36,903 rows pass
  the price rules (most of the rest trade too little).
- **Eligibility is fully explained** [3]: every ineligible row breaks at least one of the five
  rules (price below \$1: 68% of them; trading below \$10,000 a day: 69%; no SEC filing in 12
  months: 24%; this last rule no longer applies to price-only stocks here). Funds and
  blank-check companies are already left out of the file.
- **Missing values have reasons** [4]: macro and trading features are complete; price features are
  missing only where a company has less than a year of prices (at most 5%); insider and 8-K
  features are missing for foreign filers by design (9.9% of rows), and so is market value
  (18% of rows: foreign filers and companies without a usable share count); financial ratios
  are missing where the company does not report the item (R&D, gross margin, interest exist only
  for some businesses), for foreign filers (IFRS not mapped) and price-only stocks (no filings).
- **The cleaning rules hold** [5]: no foreign filer has a market value, no market value is 0 or
  above \$5 trillion, no insider sales total above \$100 billion.
- **No leakage** [10]: no window ends on or before its row; the strongest feature has |ρ| 0.79.
- **Stable relations** [8]: none of the 15 strongest features changes direction in any year.
- **Purge** [10]: before each test year it removes 12,000–15,000 rows (about 3%), leaving
  314,000–472,000 training rows.

### The answer [6]

- Median 0.36 (a typical swing of ±36% a year); 80% of rows lie between 0.18 and 0.83. A long
  right tail: 30,396 rows above 100% a year, 18 above 1,000%.
- Small companies swing more (median: micro 0.51, large 0.25); health care most (0.58), utilities
  least (0.22). 2020 stands out (the COVID crash).
- 96 rows have an answer of exactly 0: the price did not change once in 63 days (stale quotes of
  thinly traded stocks, 66 companies, 14 of the rows price-only, mostly 2012–2013). Their
  logarithm does not exist.

### The naive baseline [7]

| | 2019 | 2020 | 2021 | 2022 | all years |
|---|---|---|---|---|---|
| mean absolute error (log) | 0.35 | 0.49 | 0.29 | 0.25 | 0.30 |
| rank correlation | 0.71 | 0.40 | 0.82 | 0.86 | 0.79 |
| future within ±25% of past | 47% | 34% | 52% | 58% | 50% |

An error of 0.30 on the log scale means about 35% too high or too low. 2020 is the hard year: a
calm end of 2019 said nothing about March 2020.

**Mean reversion** [7]: fitted on windows ending before 2019, log(future) = −0.257 + 0.775 ×
log(past). A slope below 1 means volatility moves about a fifth of the way back toward normal
each quarter. This rule lowers the error in 2019 (0.340 vs 0.351), 2020 (0.431 vs 0.490) and 2021
(0.273 vs 0.288), and is slightly worse in 2022 (0.260 vs 0.254).

### What relates to the answer [8]

- **Price features dominate**: past volatility in its four forms (ρ 0.75–0.79), the fall from
  the 12-month high (−0.72), distance from the high (−0.60), price level (−0.48).
- **Then the company**: loss-making (0.47), interest coverage (−0.41), size (−0.41), R&D
  intensity (0.39), cash share of assets (0.35), share growth (0.32).
- **For the change from past to future volatility** (what the baseline misses): the stock's own
  volatility (−0.31: high falls, low rises), market volatility (SPY, −0.21), recent changes in
  2-year rates (+0.14) and VIX (+0.13), unemployment (−0.13). Macro matters for the change, not
  the level.
- Weak: insider activity (|ρ| ≤ 0.16), short volume (≤ 0.08).

### Redundant features [9]

14 pairs with |ρ| ≥ 0.9: three are the same number on two scales (`close`/`log_close`,
`addv20`/`log_addv20`, `market_cap`/`log_market_cap`); insider counts and values over the same
window (0.91–0.98); `vol_252`/`idio_vol_252` (0.98); `vol_21`/`range_vol_21` (0.91); traded value
and market value (0.91).

### Left for the validation round (named cases, no rule yet) [5]

- 250 rows (9 companies) with more than 50 billion shares. Most are real ordinary-share counts of
  foreign banks (Chile, UK), whose market value is blank; a few are typos the rules cannot see
  because the company's first count was already wrong (for example a partnership with 66.7
  billion units instead of about 350 million).
- 17 rows (2 companies) with insider buys above \$10 billion in a year: one trade the price and
  size checks cannot judge (a business development company before it was listed) and one
  implausibly large trade in a large bank.
- 51 rows with a one-month return above +1,000% and 399 with a 12-month return above +2,000%, at
  prices of \$1 or more: some real, some possibly a missing split factor.
- 96 zero answers (above): the label is pre-registered and stays; the model leaves them out.

### Advice for the volatility model (005)

- Predict log(`vol_63_future`). Levels to compare: the naive baseline, the mean-reversion line
  above (the *rule* level), a linear model, LightGBM.
- Report every test year separately; 2020 will dominate any average.
- Start with the price family, size, loss-making, interest coverage, cash share, and the macro
  changes (SPY volatility, VIX change, 2-year rate change).
- For the linear model: keep one feature of each redundant pair; cut ratios at their 1st and 99th
  percentile on the training rows (book-to-market, leverage and the returns have extreme tails);
  fill missing values with the training median and add a "was missing" flag, because missing
  financials mean something (no filings, foreign, very small). LightGBM handles missing values
  itself.

- **Price-only stocks** (V2-26) are now in training (9,577 rows with an answer), but their prices
  are thinner: 29% of their rows have no trading on more than a fifth of days, against 2.3% for
  US filers. Report the test years for them separately; if the model does worse for them, the
  app labels their forecast as less tested.